In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [3]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", base_url="http://192.168.101.78:11434")


In [22]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [23]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [24]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'Please '
                                                                          'let '
                                                                          'me '
                                                                          'know '
                                                                          'what '
                     

In [25]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John,\n\nNo problem at all. Please let me know what time works best for you tomorrow, or feel free to suggest a different day that suits your schedule.\n\nLooking forward to catching up.\n\nBest regards,\nSeán'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John,\\n\\nNo problem at all. Please let me know what time works best for you tomorrow, or feel free to suggest a different day that suits your schedule.\\n\\nLooking forward to catching up.\\n\\nBest regards,\\nSeán'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='cda160aeb9386896614c00ae5919979f')]


In [26]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem at all. Please let me know what time works best for you tomorrow, or feel free to suggest a different day that suits your schedule.

Looking forward to catching up.

Best regards,
Seán


## Approve

In [9]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='8d1fec4c-1a72-4ab3-8512-e8cbfec73d20'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-04-21T13:44:58.422176Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4506611400, 'load_duration': 3005407200, 'prompt_eval_count': 338, 'prompt_eval_duration': 194835200, 'eval_count': 90, 'eval_duration': 1163995100, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--019db049-6944-7930-ba03-bb7f3c2d7d4b-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'dcf8fb8b-ddb6-42a7-ad7f-e5156ad62ece', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 338, 'output_tokens'

## Reject

In [27]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'Please '
                                                                          'let '
                                                                          'me '
                                                                          'know '
                                                                          'what '
                     

In [31]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem at all. Please let me know what time works best for you tomorrow, or feel free to suggest a different day that suits your schedule.

Looking forward to catching up.

Your merciful leader, Seán


## Edit

In [29]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'Please '
                                                                          'let '
                                                                          'me '
                                                                          'know '
                                                                          'what '
                     